# Causal intervention test: seriousness

Steers the residual stream toward each seriousness class on everyday advice questions (questions/seriousness.txt) and checks whether the response's tone shifts toward playful (`low`) or formal (`high`). There's no ground-truth answer for tone, so this one is read qualitatively rather than auto-scored.

Method: TalkTuner's (Chen et al. 2024) activation-steering recipe -- add `n_scale * (target_one_hot @ control_probe.weight)` to the residual stream at the last token position, for a window of layers, on every generation step. See `intervention_common.py` and `docs/llama_dataset_synthesis.md`.

In [1]:
import sys
sys.path.insert(0, '.')
import json
import intervention_common as ic
import importlib
importlib.reload(ic)


<module 'intervention_common' from '/root/mats12/nb/causality_tests/./intervention_common.py'>

In [2]:
ATTRIBUTE = "seriousness"
FROM_IDX = 0  # steer decoder blocks [FROM_IDX, TO_IDX), centered on this
TO_IDX = 11      # attribute's best control-probe layer (4)
N_SCALE = 7.0  # TalkTuner's own fixed-magnitude default
BATCH_SIZE = 5
MAX_NEW_TOKENS = 200

In [3]:
tokenizer, model = ic.load_model()
probes = ic.load_control_probes(ATTRIBUTE)
layer_names = ic.which_layers(model, FROM_IDX, TO_IDX)
labels = ic.class_names(ATTRIBUTE)
print(f"classes: {labels}")
print(f"steering {len(layer_names)} layers: {layer_names}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

classes: ['low', 'medium', 'high']
steering 11 layers: ['model.layers.0', 'model.layers.1', 'model.layers.2', 'model.layers.3', 'model.layers.4', 'model.layers.5', 'model.layers.6', 'model.layers.7', 'model.layers.8', 'model.layers.9', 'model.layers.10']


In [4]:
questions = ic.load_plain_questions(ATTRIBUTE)
question_texts = questions
for q in questions:
    print(f"- {q}")

- What should I do this weekend?
- Can you help me plan a birthday party?
- What's a good way to spend a rainy afternoon?
- I have a work meeting tomorrow, any tips?
- What should I cook for dinner tonight?
- Any suggestions for a road trip playlist?
- How should I respond to a coworker's email about a missed deadline?
- What's a fun icebreaker for a team meeting?
- Give me some advice for my first day at a new job.
- What should I write in a birthday card for a close friend?


## Generate responses

Baseline (unintervened), then one steered pass per class label.

In [5]:
responses_by_condition = {}
responses_by_condition["unintervened"] = ic.generate_responses(
    model, tokenizer, question_texts, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
)

generating:   0%|          | 0/2 [00:00<?, ?it/s]

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


In [6]:
for class_idx, label in enumerate(labels):
    target = ic.one_hot(class_idx, len(labels))
    hook = ic.make_steering_hook(probes, target, n_scale=N_SCALE)
    print(f"=== steering toward '{label}' ===")
    responses_by_condition[label] = ic.generate_responses(
        model, tokenizer, question_texts, layer_names=layer_names, edit_output=hook,
        batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS,
    )

=== steering toward 'low' ===


generating:   0%|          | 0/2 [00:00<?, ?it/s]

=== steering toward 'medium' ===


generating:   0%|          | 0/2 [00:00<?, ?it/s]

=== steering toward 'high' ===


generating:   0%|          | 0/2 [00:00<?, ?it/s]

## View responses side by side

In [7]:
for i, q in enumerate(question_texts):
    print("=" * 100)
    print(q)
    print("=" * 100)
    for condition, responses in responses_by_condition.items():
        print(f"--- {condition} ---")
        print(responses[i])
        print()

What should I do this weekend?
--- unintervened ---
Hello! I'd be happy to help you decide what to do this weekend! Before I suggest any activities, may I ask what type of things you enjoy doing? Do you like outdoor activities, indoor activities, or a mix of both? Are you looking for something relaxing or something more adventurous? Additionally, do you have any specific location in mind or are you open to exploring new places? Knowing your preferences will help me provide more tailored suggestions. 😊

--- low ---
:::::::]]]] " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " " "

--- medium ---
endendendendend]]]]].]ightforwardforwardforwardforwardforwardforwardforwar

## Save transcripts + raw responses

In [8]:
config = dict(from_idx=FROM_IDX, to_idx=TO_IDX, n_scale=N_SCALE,
              batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS, labels=labels)
out_dir = ic.save_intervention_results(ATTRIBUTE, questions, responses_by_condition, config)
print(f"Saved to {out_dir}")

Saved to /root/mats12/nb/causality_tests/intervention_results/seriousness
